# LA Studio voice-isolation - Spleeter 2-stem FP16

This notebook runs exactly `sherpa-onnx-spleeter-2stems-fp16` from the declared k2-fsa artifact on the temporary **Colab GPU worker**. The LA Studio worker and launcher are embedded in this notebook; no LA Studio GitHub repository or repository token is required at runtime.

The worker performs a CUDA startup probe before it prints a URL. It also sends long audio as bounded, overlapping segments, so the Spleeter FP16 CUDA convolution plan remains within the verified shape.

1. Choose **Runtime -> Change runtime type -> GPU**.
2. Run all cells. The final cell must print `startup probe: passed`.
3. Copy the printed URL and token to Dubbing -> Colab setup, then press **Check Colab**.


In [ ]:
!nvidia-smi
%pip install -q --upgrade --no-cache-dir "onnxruntime-gpu==1.21.0" "kaldi-native-fbank" "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3" "python-multipart==0.0.20"

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No Colab CUDA GPU is available. Select Runtime > Change runtime type > GPU, then restart and Run all.')
print('Colab CUDA:', torch.cuda.get_device_name(0))

!wget -q --show-progress -O /content/spleeter.tar.bz2 https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2
!tar -xjf /content/spleeter.tar.bz2 -C /content


In [ ]:
from hashlib import sha256
from pathlib import Path

EMBEDDED_WORKERS = {
    'la_studio_separation_worker.py': ('307861926e13ff9849b04594074b573b1da063b1791c56dc2f502ab64991c5af', '"""Temporary Direct Colab worker for the exact Spleeter 2-stem FP16 artifact.\n\nThe worker deliberately uses ONNX Runtime\'s CUDA provider directly.  The\nsherpa-onnx source-separation wrapper fixes its CUDA convolution search to\nHEURISTIC, which fails on some current Colab cuDNN 9 images.  This worker uses\nthe same upstream FP16 ONNX files, but chooses ORT\'s documented DEFAULT\nconvolution algorithm instead and bounds every inference input.\n"""\n\nimport math\nimport os\nimport secrets\nimport shutil\nimport subprocess\nimport threading\nimport traceback\nfrom pathlib import Path\n\nimport kaldi_native_fbank as knf\nimport numpy as np\nimport torch\nimport onnxruntime as ort\nimport soundfile as sf\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\nfrom fastapi.responses import FileResponse\n\n\nWORKER_CONTRACT = "spleeter-cuda-safe-20260816.1"\nMODEL_ID = "sherpa-onnx-spleeter-2stems-fp16"\nMODEL_NAME = "Spleeter 2-stem FP16"\nUPSTREAM_MODEL = "k2-fsa/sherpa-onnx-spleeter-2stems-fp16"\nARTIFACT_URL = (\n    "https://github.com/k2-fsa/sherpa-onnx/releases/download/"\n    "source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2"\n)\nMODEL_ROOT = Path("/content/sherpa-onnx-spleeter-2stems-fp16")\nTOKEN = os.environ["LA_STUDIO_COLAB_SEPARATION_TOKEN"]\nROOT = Path("/content/la-studio-separation-jobs") / MODEL_ID\nROOT.mkdir(parents=True, exist_ok=True)\n\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\nMAX_AUDIO_SECONDS = 30 * 60\nARTIFACT_TTL_SECONDS = 1800\n# A 20-second core has at most two 512-frame Spleeter splits.  The previous\n# worker sent a complete long video through one CUDA call, producing a large\n# dynamic Conv shape that caused CUDNN_FE_HEURISTIC_QUERY_FAILED on Colab.\nCORE_SECONDS = 20.0\nCONTEXT_SECONDS = 1.5\nPROBE_SECONDS = CORE_SECONDS\n\nALLOWED_CONTENT_TYPES = {\n    "audio/wav", "audio/x-wav", "audio/mpeg", "audio/mp4", "audio/webm",\n    "audio/ogg", "audio/flac", "video/mp4", "video/webm", "video/quicktime",\n    "video/x-matroska", "application/octet-stream",\n}\nALLOWED_EXTENSIONS = {\n    ".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac", ".mkv", ".mov", ".avi",\n}\nCUDA_OPTIONS = {\n    # DEFAULT avoids the cuDNN heuristic-plan query that failed in the old\n    # sherpa-onnx wrapper.  The exact model remains FP16 and executes on CUDA.\n    "cudnn_conv_algo_search": "DEFAULT",\n    "cudnn_conv_use_max_workspace": "1",\n    "do_copy_in_default_stream": "1",\n    "arena_extend_strategy": "kSameAsRequested",\n}\n\nif not torch.cuda.is_available():\n    raise RuntimeError("Colab GPU is not available; select a GPU runtime before starting this worker")\n\n\ndef _cuda_session(path: Path) -> ort.InferenceSession:\n    if "CUDAExecutionProvider" not in ort.get_available_providers():\n        raise RuntimeError("ONNX Runtime CUDAExecutionProvider is unavailable in this Colab runtime")\n    options = ort.SessionOptions()\n    options.intra_op_num_threads = 1\n    options.inter_op_num_threads = 1\n    session = ort.InferenceSession(\n        str(path),\n        sess_options=options,\n        providers=[("CUDAExecutionProvider", CUDA_OPTIONS), "CPUExecutionProvider"],\n    )\n    if not session.get_providers() or session.get_providers()[0] != "CUDAExecutionProvider":\n        raise RuntimeError("The exact Spleeter ONNX session did not bind CUDAExecutionProvider")\n    return session\n\n\nclass ExactSpleeterCuda:\n    def __init__(self) -> None:\n        vocals = MODEL_ROOT / "vocals.fp16.onnx"\n        accompaniment = MODEL_ROOT / "accompaniment.fp16.onnx"\n        if not vocals.is_file() or not accompaniment.is_file():\n            raise RuntimeError("The exact Spleeter FP16 ONNX artifacts are missing")\n        self.vocals = _cuda_session(vocals)\n        self.accompaniment = _cuda_session(accompaniment)\n        self.stft_config = knf.StftConfig(\n            n_fft=4096,\n            hop_length=1024,\n            win_length=4096,\n            center=False,\n            window_type="hann",\n        )\n\n    @staticmethod\n    def _stft(samples: np.ndarray, channel: int) -> tuple[np.ndarray, np.ndarray]:\n        result = knf.Stft(knf.StftConfig(\n            n_fft=4096, hop_length=1024, win_length=4096,\n            center=False, window_type="hann",\n        ))(samples[:, channel].tolist())\n        real = np.asarray(result.real, dtype=np.float32).reshape(result.num_frames, -1)\n        imag = np.asarray(result.imag, dtype=np.float32).reshape(result.num_frames, -1)\n        return real, imag\n\n    def process(self, sample_rate: int, samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n        if sample_rate != 44100:\n            raise RuntimeError(f"expected 44100 Hz worker input, received {sample_rate}")\n        if samples.ndim != 2 or samples.shape[1] != 2 or samples.shape[0] == 0:\n            raise RuntimeError("expected non-empty stereo audio")\n        real0, imag0 = self._stft(samples, 0)\n        real1, imag1 = self._stft(samples, 1)\n        if real0.shape[0] == 0 or real1.shape[0] == 0:\n            raise RuntimeError("audio is too short for Spleeter analysis")\n        frame_count = real0.shape[0]\n        if real1.shape[0] != frame_count:\n            raise RuntimeError("stereo channel frame counts differ")\n\n        magnitude0 = np.sqrt(real0[:, :1024] ** 2 + imag0[:, :1024] ** 2).astype(np.float32)\n        magnitude1 = np.sqrt(real1[:, :1024] ** 2 + imag1[:, :1024] ** 2).astype(np.float32)\n        padded_frames = int(math.ceil(frame_count / 512.0) * 512)\n        if padded_frames != frame_count:\n            padding = ((0, padded_frames - frame_count), (0, 0))\n            magnitude0 = np.pad(magnitude0, padding)\n            magnitude1 = np.pad(magnitude1, padding)\n        model_input = np.ascontiguousarray(\n            np.stack((magnitude0, magnitude1), axis=0).reshape(2, -1, 512, 1024),\n            dtype=np.float32,\n        )\n        vocals_spec = self.vocals.run(None, {self.vocals.get_inputs()[0].name: model_input})[0]\n        accompaniment_spec = self.accompaniment.run(\n            None, {self.accompaniment.get_inputs()[0].name: model_input}\n        )[0]\n        denominator = vocals_spec ** 2 + accompaniment_spec ** 2 + 1e-10\n        masks = (\n            (vocals_spec ** 2 + 5e-11) / denominator,\n            (accompaniment_spec ** 2 + 5e-11) / denominator,\n        )\n\n        stems: list[np.ndarray] = []\n        for mask in masks:\n            channels: list[np.ndarray] = []\n            for channel, (real, imag) in enumerate(((real0, imag0), (real1, imag1))):\n                channel_mask = mask[channel].reshape(-1, 1024)[:frame_count]\n                channel_mask = np.pad(channel_mask, ((0, 0), (0, real.shape[1] - 1024)))\n                masked = knf.StftResult(\n                    real=(channel_mask * real).reshape(-1).tolist(),\n                    imag=(channel_mask * imag).reshape(-1).tolist(),\n                    num_frames=frame_count,\n                )\n                waveform = knf.IStft(self.stft_config)(masked)\n                channels.append(np.asarray(waveform, dtype=np.float32))\n            stem = np.column_stack(channels)\n            stems.append(stem)\n        return stems[0], stems[1]\n\n\n# Constructing and running the same bounded shape before /health is exposed\n# proves CUDA works for this exact model.  An unsupported Colab image fails in\n# the notebook cell, rather than accepting a URL and later failing at a random\n# workflow step.\nSEPARATOR = ExactSpleeterCuda()\n_probe = np.zeros((int(44100 * PROBE_SECONDS), 2), dtype=np.float32)\n_probe_vocals, _probe_background = SEPARATOR.process(44100, _probe)\nif _probe_vocals.shape[0] == 0 or _probe_background.shape[0] == 0:\n    raise RuntimeError("exact Spleeter CUDA startup probe produced empty audio")\ndel _probe, _probe_vocals, _probe_background\n\nJOB_SLOTS = threading.BoundedSemaphore(1)\nJOB_LOCK = threading.Lock()\nJOBS: dict[str, dict] = {}\n\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=(f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. "\n                    "Open the notebook for the selected model."),\n        )\n\n\ndef media_duration_seconds(path: Path) -> float:\n    probe = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "default=nokey=1:noprint_wrappers=1", str(path)],\n        text=True, capture_output=True,\n    )\n    try:\n        duration = float(probe.stdout.strip())\n    except ValueError:\n        duration = 0.0\n    if probe.returncode != 0 or duration <= 0.0:\n        raise HTTPException(status_code=415, detail="media is unsupported or could not be decoded")\n    return duration\n\n\ndef update(job_id: str, **values) -> dict:\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n        job.update(values)\n        JOBS[job_id] = job\n        return job\n\n\ndef is_cancelled(job_id: str) -> bool:\n    with JOB_LOCK:\n        return bool(JOBS.get(job_id, {}).get("cancel_requested", False))\n\n\ndef cleanup(job_id: str) -> None:\n    with JOB_LOCK:\n        job = JOBS.pop(job_id, None)\n    if job:\n        shutil.rmtree(job.get("directory", ""), ignore_errors=True)\n\n\ndef _fit_piece(stem: np.ndarray, start: int, length: int) -> np.ndarray:\n    piece = stem[start:start + length]\n    if piece.shape[0] >= length:\n        return piece[:length]\n    return np.pad(piece, ((0, length - piece.shape[0]), (0, 0)))\n\n\ndef separate_bounded(job_id: str, samples: np.ndarray, sample_rate: int) -> tuple[np.ndarray, np.ndarray]:\n    core = int(CORE_SECONDS * sample_rate)\n    context = int(CONTEXT_SECONDS * sample_rate)\n    total = samples.shape[0]\n    pieces = max(1, math.ceil(total / core))\n    vocals_parts: list[np.ndarray] = []\n    background_parts: list[np.ndarray] = []\n    for index, core_start in enumerate(range(0, total, core), start=1):\n        if is_cancelled(job_id):\n            raise RuntimeError("Separation cancelled")\n        core_end = min(total, core_start + core)\n        window_start = max(0, core_start - context)\n        window_end = min(total, core_end + context)\n        update(\n            job_id,\n            status="running",\n            progress=20 + int(65 * (index - 1) / pieces),\n            detail=f"{MODEL_NAME} CUDA segment {index}/{pieces}",\n        )\n        vocals, background = SEPARATOR.process(sample_rate, samples[window_start:window_end])\n        trim = core_start - window_start\n        core_length = core_end - core_start\n        vocals_parts.append(_fit_piece(vocals, trim, core_length))\n        background_parts.append(_fit_piece(background, trim, core_length))\n        update(\n            job_id,\n            status="running",\n            progress=20 + int(65 * index / pieces),\n            detail=f"{MODEL_NAME} CUDA segment {index}/{pieces} complete",\n        )\n    return np.concatenate(vocals_parts, axis=0), np.concatenate(background_parts, axis=0)\n\n\ndef concise_failure(error: Exception) -> str:\n    text = str(error).replace("\\n", " ").strip()\n    if "CUDNN" in text.upper() or "CUDA" in text.upper():\n        return ("The verified Colab CUDA worker failed during Spleeter inference. "\n                "No local model was started. Stop this job, reopen the current Spleeter notebook, "\n                "and use its startup probe before reconnecting. Full worker detail is in the Colab output.")\n    return f"{type(error).__name__}: {text[:600]}"\n\n\ndef run_job(job_id: str, directory: Path, source: Path, output_format: str) -> None:\n    try:\n        update(job_id, status="running", progress=12, detail="Decoding media for bounded CUDA separation")\n        wav_path = directory / "source-44100-stereo.wav"\n        subprocess.run(\n            ["ffmpeg", "-y", "-v", "error", "-i", str(source), "-vn", "-acodec", "pcm_s16le",\n             "-ar", "44100", "-ac", "2", str(wav_path)],\n            check=True,\n        )\n        samples, sample_rate = sf.read(wav_path, dtype="float32", always_2d=True)\n        samples = np.ascontiguousarray(samples, dtype=np.float32)\n        vocals_data, background_data = separate_bounded(job_id, samples, sample_rate)\n        if is_cancelled(job_id):\n            update(job_id, status="cancelled", progress=0, detail="Separation cancelled")\n            return\n        update(job_id, status="running", progress=90, detail="Writing separated CUDA stems")\n        suffix = ".wav" if output_format == "wav" else ".flac"\n        vocals = directory / ("vocals" + suffix)\n        background = directory / ("background" + suffix)\n        # FLAC is lossless and typically reduces the 44.1 kHz stereo transfer\n        # by far more than 50%; PCM WAV remains the explicit compatibility\n        # choice for an operator who needs it.\n        sf.write(vocals, vocals_data, sample_rate,\n                 format="WAV" if output_format == "wav" else "FLAC",\n                 subtype="PCM_16")\n        sf.write(background, background_data, sample_rate,\n                 format="WAV" if output_format == "wav" else "FLAC",\n                 subtype="PCM_16")\n        update(\n            job_id, status="ready", progress=100, detail="Separated CUDA stems are ready",\n            vocals=str(vocals), background=str(background),\n            artifact_format=output_format, artifacts_ready=True,\n        )\n    except Exception as error:\n        traceback.print_exc()\n        update(job_id, status="failed", progress=0, detail=concise_failure(error))\n    finally:\n        threading.Timer(ARTIFACT_TTL_SECONDS, cleanup, args=[job_id]).start()\n        JOB_SLOTS.release()\n\n\napp = FastAPI(title=f"LA Studio Voice Isolation - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "onnxruntime": ort.__version__,\n        "cuda_provider_options": CUDA_OPTIONS,\n        "bounded_core_seconds": CORE_SECONDS,\n        "startup_probe": "passed",\n        "cpu_fallback": False,\n    }\n\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "voice-isolation",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "artifact_url": ARTIFACT_URL,\n                "stems": ["vocals", "background"],\n                "formats": ["flac", "wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n\n@app.post("/v1/audio/separations")\nasync def create_separation(\n    file: UploadFile = File(...),\n    stems: str = Form("vocals,background"),\n    model: str = Form(...),\n    output_format: str = Form("flac"),\n    authorization: str | None = Header(default=None),\n):\n    authorize(authorization)\n    require_exact_model(model)\n    if stems != "vocals,background":\n        raise HTTPException(status_code=422, detail="this worker returns vocals and background stems")\n    output_format = output_format.strip().lower()\n    if output_format not in {"flac", "wav"}:\n        raise HTTPException(status_code=422, detail="output_format must be flac or wav")\n    suffix = Path(file.filename or "source.wav").suffix.lower() or ".wav"\n    if suffix not in ALLOWED_EXTENSIONS:\n        raise HTTPException(status_code=415, detail="unsupported media filename extension")\n    if file.content_type and file.content_type not in ALLOWED_CONTENT_TYPES:\n        raise HTTPException(status_code=415, detail="unsupported media MIME type")\n    if not JOB_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab separation worker is busy; retry shortly")\n    job_id = secrets.token_urlsafe(18)\n    directory = ROOT / job_id\n    directory.mkdir(parents=True, exist_ok=True)\n    source = directory / ("source" + suffix)\n    try:\n        with source.open("wb") as output:\n            while chunk := await file.read(1024 * 1024):\n                output.write(chunk)\n                if output.tell() > MAX_UPLOAD_BYTES:\n                    raise HTTPException(status_code=413, detail="media exceeds 512 MB upload limit")\n        if source.stat().st_size <= 0:\n            raise HTTPException(status_code=413, detail="media must not be empty")\n        if media_duration_seconds(source) > MAX_AUDIO_SECONDS:\n            raise HTTPException(status_code=413, detail="media exceeds the 30 minute duration limit")\n    except Exception:\n        shutil.rmtree(directory, ignore_errors=True)\n        JOB_SLOTS.release()\n        raise\n    finally:\n        await file.close()\n    update(\n        job_id, status="queued", progress=10,\n        detail=f"Media uploaded; {MODEL_NAME} CUDA job is queued",\n        directory=str(directory), cancel_requested=False,\n    )\n    threading.Thread(target=run_job, args=(job_id, directory, source, output_format), daemon=True).start()\n    return {"job_id": job_id, "status": "queued", "progress": 10,\n            "artifact_format": output_format}\n\n\n@app.get("/v1/audio/separations/{job_id}")\ndef separation_status(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n    if not job:\n        raise HTTPException(status_code=404, detail="separation job not found")\n    return {key: job.get(key) for key in ("status", "progress", "detail", "artifact_format", "artifacts_ready") if key in job} | {"job_id": job_id}\n\n\n@app.get("/v1/audio/separations/{job_id}/artifacts/{stem}")\ndef artifact(job_id: str, stem: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if stem not in {"vocals", "background"}:\n        raise HTTPException(status_code=404, detail="unknown stem")\n    with JOB_LOCK:\n        job = dict(JOBS.get(job_id, {}))\n    path = Path(job.get(stem, ""))\n    if job.get("status") != "ready" or not path.is_file():\n        raise HTTPException(status_code=409, detail="stem is not ready")\n    output_format = job.get("artifact_format", "wav")\n    media_type = "audio/flac" if output_format == "flac" else "audio/wav"\n    return FileResponse(path, media_type=media_type, filename=stem + "." + output_format)\n\n\n@app.delete("/v1/audio/separations/{job_id}")\ndef cancel_separation(job_id: str, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    with JOB_LOCK:\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail="separation job not found")\n        JOBS[job_id]["cancel_requested"] = True\n        JOBS[job_id]["status"] = "cancelling"\n    return {"job_id": job_id, "status": "cancelling"}\n'),
    'la_studio_separation_launcher.py': ('9ca893f8e06826bb875e68a7e364cdb43be515882b8438f01eb71465874375d1', '"""Launch the exact Spleeter Direct Colab worker and a temporary tunnel."""\n\nimport json\nimport os\nimport re\nimport secrets\nimport socket\nimport subprocess\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\n\nWORKER_CONTRACT = "spleeter-cuda-safe-20260816.1"\nMODEL_ID = "sherpa-onnx-spleeter-2stems-fp16"\nCAPABILITY_LABEL = "Voice Isolation"\nPORT = 3924\nTOKEN_ENV = "LA_STUDIO_COLAB_SEPARATION_TOKEN"\nURL_ENV = "LA_STUDIO_COLAB_SEPARATION_URL"\nMODEL_ENV = "LA_STUDIO_COLAB_SEPARATION_MODEL"\nWORKER_LOG = Path("/content/la_studio_separation_worker.log")\nTUNNEL_LOG = Path("/content/la_studio_separation_tunnel.log")\nSTARTUP_TIMEOUT_SECONDS = 20 * 60\nTUNNEL_TIMEOUT_SECONDS = 90\nPUBLIC_TUNNEL_VERIFY_TIMEOUT_SECONDS = 120\n\n\ndef port_is_occupied(port: int) -> bool:\n    try:\n        with socket.create_connection(("127.0.0.1", port), timeout=0.5):\n            return True\n    except OSError:\n        return False\n\n\ndef tail(path: Path, limit: int = 12000) -> str:\n    try:\n        return path.read_text(encoding="utf-8", errors="replace")[-limit:]\n    except FileNotFoundError:\n        return "(log was not created)"\n\n\ndef stop(process: subprocess.Popen | None) -> None:\n    if process is None or process.poll() is not None:\n        return\n    process.terminate()\n    try:\n        process.wait(timeout=10)\n    except subprocess.TimeoutExpired:\n        process.kill()\n\n\ndef cloudflared_ready() -> bool:\n    try:\n        return subprocess.run(\n            ["cloudflared", "--version"], stdout=subprocess.DEVNULL,\n            stderr=subprocess.DEVNULL, check=False,\n        ).returncode == 0\n    except OSError:\n        # A fresh Colab runtime normally has no cloudflared executable yet.\n        # subprocess.run raises FileNotFoundError in that case rather than\n        # returning a non-zero status.\n        return False\n\n\ndef ensure_cloudflared() -> None:\n    if cloudflared_ready():\n        return\n    package_path = "/content/la-studio-cloudflared.deb"\n    download = subprocess.run(\n        ["curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",\n         "--output", package_path,\n         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],\n        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,\n    )\n    if download.returncode != 0:\n        raise RuntimeError("Could not download cloudflared: " + (download.stdout[-1200:] or "no output"))\n    install = subprocess.run(["dpkg", "-i", package_path], text=True,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)\n    if install.returncode != 0 or not cloudflared_ready():\n        raise RuntimeError("Could not install cloudflared: " + (install.stdout[-1200:] or "no output"))\n\n\ndef verify_public_tunnel(public_url: str, token: str) -> tuple[bool, str]:\n    """Attempt a Colab-side check of the public hostname.\n\n    This is useful diagnostic evidence, but it is not authoritative: a\n    Colab runtime can fail to resolve a newly-created trycloudflare hostname\n    even while the desktop can reach it.  The desktop\'s Check Colab action is\n    the authoritative authenticated capability/model verification.\n    """\n    try:\n        request = urllib.request.Request(\n            public_url.rstrip("/") + "/health",\n            headers={"Authorization": "Bearer " + token},\n        )\n        with urllib.request.urlopen(request, timeout=12) as response:\n            health = json.loads(response.read().decode("utf-8"))\n        if (response.status == 200 and health.get("ready") is True\n                and str(health.get("device", "")).lower() == "cuda"\n                and str(health.get("model", "")).strip().lower() == MODEL_ID\n                and health.get("cpu_fallback") is False\n                and health.get("startup_probe") == "passed"):\n            return True, "verified"\n        return False, "unexpected public /health response: " + json.dumps(health, ensure_ascii=False)\n    except urllib.error.HTTPError as error:\n        return False, f"public /health returned HTTP {error.code}: " + error.read().decode(\n            "utf-8", errors="replace"\n        )[:1000]\n    except Exception as error:\n        return False, f"public /health is not reachable: {type(error).__name__}: {error}"\n\n\nif port_is_occupied(PORT):\n    raise RuntimeError(\n        f"Port {PORT} is occupied by an earlier Colab worker. Use Runtime > Disconnect and delete runtime, "\n        "then Run all once for this exact model."\n    )\n\ntoken = secrets.token_urlsafe(32)\nenvironment = os.environ.copy()\nenvironment[TOKEN_ENV] = token\nenvironment["PYTHONUNBUFFERED"] = "1"\nwith WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:\n    worker = subprocess.Popen(\n        [sys.executable, "-m", "uvicorn", "la_studio_separation_worker:app",\n         "--host", "127.0.0.1", "--port", str(PORT)],\n        cwd="/content", env=environment, stdout=worker_output, stderr=subprocess.STDOUT,\n        start_new_session=True,\n    )\n\nprint(f"Starting exact CUDA {CAPABILITY_LABEL} worker; it must pass its bounded Spleeter startup probe.")\ndeadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS\nlast_error = "worker has not answered /health yet"\nwhile time.monotonic() < deadline:\n    if worker.poll() is not None:\n        raise RuntimeError(\n            f"The exact-model worker exited before becoming CUDA-ready (exit code {worker.returncode}).\\n\\n"\n            "---- worker log ----\\n" + tail(WORKER_LOG)\n        )\n    try:\n        request = urllib.request.Request(f"http://127.0.0.1:{PORT}/health",\n                                         headers={"Authorization": "Bearer " + token})\n        with urllib.request.urlopen(request, timeout=10) as response:\n            health = json.loads(response.read().decode("utf-8"))\n        if (response.status == 200 and health.get("ready") is True\n                and str(health.get("device", "")).lower() == "cuda"\n                and str(health.get("model", "")).strip().lower() == MODEL_ID\n                and health.get("cpu_fallback") is False\n                and health.get("startup_probe") == "passed"):\n            print("Exact CUDA worker passed startup probe:", health)\n            break\n        last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)\n    except urllib.error.HTTPError as error:\n        last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]\n    except Exception as error:\n        last_error = f"/health is not ready: {type(error).__name__}: {error}"\n    time.sleep(2)\nelse:\n    stop(worker)\n    raise RuntimeError(\n        f"The exact-model worker did not become CUDA-ready within {STARTUP_TIMEOUT_SECONDS // 60} minutes. "\n        f"Last check: {last_error}\\n\\n---- worker log ----\\n" + tail(WORKER_LOG)\n    )\n\nensure_cloudflared()\nwith TUNNEL_LOG.open("w", encoding="utf-8", buffering=1) as tunnel_output:\n    tunnel = subprocess.Popen(\n        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],\n        stdout=tunnel_output, stderr=subprocess.STDOUT, start_new_session=True,\n    )\n\npublic_url = ""\npublic_tunnel_verified = False\ncandidate_url = ""\nlast_tunnel_error = "cloudflared has not published a public URL yet"\ndeadline = time.monotonic() + max(TUNNEL_TIMEOUT_SECONDS, PUBLIC_TUNNEL_VERIFY_TIMEOUT_SECONDS)\nwhile time.monotonic() < deadline and not public_url:\n    if tunnel.poll() is not None:\n        last_tunnel_error = f"cloudflared exited with code {tunnel.returncode}"\n        break\n    match = re.search(r"https://[^\\s\\"\']+\\.trycloudflare\\.com", tail(TUNNEL_LOG, 4000))\n    if match:\n        candidate_url = match.group(0)\n        verified, last_tunnel_error = verify_public_tunnel(candidate_url, token)\n        if verified:\n            public_url = candidate_url\n            public_tunnel_verified = True\n            break\n    time.sleep(2)\n\nif not public_url:\n    # A Quick Tunnel may be healthy from the desktop even when this Colab\n    # runtime cannot resolve its just-created DNS name.  The exact CUDA worker\n    # was already verified locally above.  Preserve a candidate only for this\n    # narrow DNS/connectivity failure, and let the desktop prove the public\n    # endpoint before it can run any job.  Never publish a candidate after an\n    # HTTP/auth/model-contract mismatch, or after either local process exited.\n    if (candidate_url\n            and last_tunnel_error.startswith("public /health is not reachable:")\n            and worker.poll() is None\n            and tunnel.poll() is None):\n        public_url = candidate_url\n    else:\n        stop(tunnel)\n        stop(worker)\n        raise RuntimeError(\n            "cloudflared did not create a verified public trycloudflare endpoint within "\n            f"{max(TUNNEL_TIMEOUT_SECONDS, PUBLIC_TUNNEL_VERIFY_TIMEOUT_SECONDS)} seconds. "\n            f"Last check: {last_tunnel_error}\\n"\n            "---- cloudflared log ----\\n" + tail(TUNNEL_LOG, 4000)\n        )\n\nprint("\\nLA Studio exact-model Colab worker is ready")\nif public_tunnel_verified:\n    print("Verified the public Cloudflare tunnel against this exact CUDA worker.")\nelse:\n    print(\n        "Cloudflare emitted a public endpoint, but this Colab runtime could not "\n        "complete its own DNS/public-health check. The desktop Check Colab action "\n        "must now verify the public endpoint, bearer token, capability, and exact model."\n    )\nprint(URL_ENV + "=" + public_url)\nprint(TOKEN_ENV + "=" + token)\nprint(MODEL_ENV + "=" + MODEL_ID)\nprint("Click Check Colab in the matching LA Studio feature before running it.")\n')
}
for destination, (expected_sha256, source) in EMBEDDED_WORKERS.items():
    actual_sha256 = sha256(source.encode('utf-8')).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f'Embedded worker integrity check failed for {destination}: {actual_sha256}')
    Path('/content', destination).write_text(source, encoding='utf-8')
print('Embedded verified exact-model CUDA worker and launcher templates.')


In [ ]:
!python /content/la_studio_separation_launcher.py
